# 🔥 IGNIS-μ · Entrega interactiva al ingeniero de software 2
**10 celdas, 4 ejecutables.** Inferencia BN-01/BN-02, CPT condicional BN-03 y sensibilidad contrafactual CF-01. Los valores son del notebook PGM(2), no observaciones generadas. No se atribuyen efectos físicos `do()` identificados.

## 1. Localización del paquete
Abre este notebook desde la carpeta descomprimida `IGNIS_mu_Interactive_Handoff`. El motor es autocontenido (biblioteca estándar). Los endpoints requieren el FastAPI existente.

In [ ]:
from pathlib import Path
import sys, json
RAIZ_PAQUETE = Path.cwd().resolve()
if not (RAIZ_PAQUETE / 'install_into_repo.py').is_file():
    raise FileNotFoundError('Abre el notebook desde la carpeta IGNIS_mu_Interactive_Handoff.')
sys.path.insert(0, str(RAIZ_PAQUETE))
from ignis_interactive.engine import infer, counterfactual, schema
print('✓ Modelos:', list(schema()['modelos']))

## 2. Inferencia matemática en vivo (sin API)
El snapshot incluido utiliza la ejecución de PGM(2): BN-01 reconstruida desde conteos, BN-02/BN-03 y CF-01 redondeadas desde salidas del notebook. El instalador prioriza JSON originales para máxima precisión.

In [ ]:
bn = infer('BN01','PMT3',{'PatronMFC':'AMBOS_POSITIVOS','Radiometro':'ALTO'})
cf = counterfactual(day='20176',from_a=0,to_a=1)
assert abs(bn['distribucion']['ALTO'] - 0.02733211) < 1e-7
assert cf['efecto_causal_identificado'] is False
print('P(PMT3=ALTO|MFC=AMBOS_POSITIVOS,Radiometro=ALTO):',bn['distribucion']['ALTO'])
for m in cf['modelos']:
    print(m['modelo'], m['probabilidad_respuesta_alternativa_mayor'])
print('Cotas por jornada 20176:',cf['cotas_acoplamiento'])

## 3. Integrar en FastAPI + Angular
El instalador crea respaldos y agrega el router y componente Angular al repositorio local sin tocar Docker, Supabase ni los modelos 3D. Para habilitar predicciones individuales, localizar la carpeta original `IGNIS_mu_API_Export` que contiene `03_CONTRAFACTUALES_853B.json` de 106 registros.

In [ ]:
from install_into_repo import install,find_repo
REPO=Path.home()/'Escritorio'/'Hackaton NASA'/'fire-hunters-project'
if not (REPO/'back/ignis_science/ignis_router.py').is_file():
    REPO=find_repo(Path.cwd())
EXPORTS=REPO/'IGNIS_mu_API_Export'
resultado=install(REPO,EXPORTS if EXPORTS.is_dir() else None)
print(json.dumps(resultado,indent=2,ensure_ascii=False))

## 4. Verificar la API después de reiniciar el contenedor
Desde la raíz del repositorio: `docker compose restart spacecraft-node`. Angular debe estar ejecutándose con `npm start`; el componente aparece antes del footer. La prueba HTTP no cambia datos.

In [ ]:
from urllib.request import urlopen,Request
API='http://127.0.0.1:8000'
for ruta in ['/ignis/interactive/schema','/ignis/interactive/trials']:
    try:
        with urlopen(API+ruta,timeout=5) as r:
            print('✓',ruta,'HTTP',r.status)
    except Exception as exc: print('⚠',ruta,exc)
req=Request(API+'/ignis/interactive/inference',data=json.dumps({'modelo':'BN01','variable_objetivo':'PMT3','evidencia':{'PatronMFC':'AMBOS_POSITIVOS','Radiometro':'ALTO'}}).encode(),headers={'Content-Type':'application/json'},method='POST')
try:
    with urlopen(req,timeout=5) as r: print('✓ POST inferencia',json.load(r)['distribucion'])
except Exception as exc: print('⚠ FastAPI pendiente:',exc)

## Guía de integración y límites científicos
- Las barras BN se recalculan al cambiar **estados discretos**; no controlar gravedad, presión u oxígeno del 3D con redes que no fueron entrenadas para esas variables.
- Las barras contrafactuales son **probabilidades de diferencias poblacionales bajo M1/M2/M3**, no la probabilidad causal individual de un ensayo. Se muestran cotas observacionales y modelos alternativos.
- El ensayo individual requiere el export original `03_CONTRAFACTUALES_853B.json`; el SCM aditivo también requiere `03_PARAMETROS_SCM_853A.json`. Si falta, el sistema informa ausencia de evidencia.
- Los 129 registros NASA BASS-II siguen expuestos en `/ignis/bassii`; no imputar velocidades físicas sin seguimiento espacial calibrado.
- Revisar el `README.md` del paquete para contratos y pruebas. **Estado:** inferencia observacional interactiva ✅ · sensibilidad contrafactual exploratoria ✅ · efecto causal físico identificado ❌.